# EDA · Tanzania · low birth weight

Dataset: `data/processed/tz_lbw.csv`.
Outcome: low birth weight — derived from `birth_weight_g < 2500`.

Weighed live births only (skews toward facility births — note this as a limitation).

What this notebook should cover:
- shape and dtypes; confirm the row count matches the proposal
- missingness (expect blanks in told_high_bp, preceding_interval_m)
- the outcome rate
- distributions of the predictors (maternal_age, prior_pregnancies, prior_live_births, prior_stillbirth, prior_loss, preceding_interval_m, multiple, education, wealth_quintile, residence, region, told_high_bp) and the outcome rate within each category

Clear outputs before committing.


In [ ]:
from IPython.display import display, Markdown

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../../../data/processed/tz_lbw.csv")

outcome = "low_birth_weight"
predictors = ["maternal_age", "education", "wealth_quintile", "residence",
              "prior_live_births", "prior_stillbirth", "prior_loss",
              "preceding_interval_m", "multiple",
              "anc_visits", "first_anc_month", "bp_measured",
              "urine_tested", "blood_tested", "iron_tablets"]
helpers = ["weight", "v001"]
eda_only = ["region", "told_high_bp"]

df = df[[outcome] + predictors + helpers + eda_only]
df.shape

## Loading the data

The Tanzania low birth weight data is loaded, keeping only the  reccomended columns.

- #Outcome:low_birth_weight which shows whether the baby was born under 2.5 kg.
- #Predictors:
mother's age, education, wealth, place of residence, past births, gap since the last birth, twins, and antenatal care (number of visits, when the first visit happened, and whether BP, urine and blood were checked and iron tablets given).
- Weight: used to get correct rates from the survey.
- v001:the cluster number, used later to split the data.
- Region and told_high_bp:looked at in the EDA only, not used in the model.

All other columns are dropped, including information only known after the birth, such as birth weight and place of delivery.

In [ ]:
df.dtypes

This lists each column with its type. int64 and float64 mean numbers, object means text.

In [ ]:
df.isna().sum().sort_values(ascending=False)

## Missing values

preceding_interval_m has 1,030 missing values (about 22%). These are first births, so there is no previous birth to measure a gap from.
The antenatal care columns have only a few missing values: anc_visits (17), iron_tablets (9), first_anc_month (4), bp_measured(3), urine_tested(2) and blood_tested(1).
- All other columns are complete.

In [ ]:
df["told_high_bp"].value_counts(dropna=False)

#value count
The column only has 0 and 1 values: 4,330 mothers (0) and 296 mothers (1), so about 6.4% were told they had high blood pressure.

# low Birth Weight
This shows three things:

How many babies are LBW (1) and not LBW (0)
The plain rate
The weighted rate, which is the one to report, since the survey weights correct for groups being over- or under-sampled

In [ ]:
def wrate(d):
    return (d["low_birth_weight"] * d["weight"]).sum() / d["weight"].sum()

display(df["low_birth_weight"].value_counts())
display(round(df["low_birth_weight"].mean(), 3))
display(round(wrate(df), 3))

# findings 
355 of 4,626 babies (7.7%) were low birth weight. The weighted rate is also 7.7%, so the survey weights make little difference overall.

Because only about 1 in 13 babies is low birth weight, the outcome is imbalanced. This will matter when building and judging the models, since a model could score 92% accuracy just by predicting "not LBW" for every baby.

# LBW rate by category

In [ ]:
cat_cols = ["education", "wealth_quintile", "residence", "prior_stillbirth",
            "prior_loss", "multiple", "bp_measured", "urine_tested",
            "blood_tested", "iron_tablets", "region", "told_high_bp"]

def lbw_summary(col):
    s = df.groupby(col, dropna=False).apply(
        lambda g: pd.Series({
            "births": len(g),
            "share_%": round(100 * len(g) / len(df), 1),
            "lbw_rate_%": round(100 * wrate(g), 1),
        })
    )
    s["births"] = s["births"].astype(int)
    return s

overall = 100 * wrate(df)

for col in cat_cols:
    s = lbw_summary(col)
    display(Markdown(f"### {col}"))
    display(s)

    ax = s["lbw_rate_%"].plot(kind="barh", figsize=(6, 0.4 * len(s) + 1))
    ax.axvline(overall, linestyle="--", color="grey", label=f"Overall {overall:.1f}%")
    ax.set_xlabel("Weighted LBW rate (%)")
    ax.set_ylabel("")
    ax.legend()
    plt.show()

## Summary: LBW rate by category

The overall weighted LBW rate is 7.7% (355 of 4,626 babies).

### Strongest pattern
- Multiple births: 56.3% of twins were low birth weight, compared with 6.8% of single births. This is by far the biggest difference in the data, although twins are only 85 births (1.8%).

### Moderate patterns
- Told high blood pressure: 12.1% LBW among mothers told they had high blood pressure, compared with 7.5% among those who were not (296 mothers, 6.4%). 
- Residence: urban 9.0% vs rural 7.1%. This is unexpected and may be because urban births are more likely to be weighed.
- Prior stillbirth: 9.2% vs 7.6%, but only 172 mothers had a previous stillbirth.
- Prior loss: 8.7% vs 7.6%.

### Little or no clear pattern
- Education: rates are similar across all levels (7.4–8.2%). Only 61 mothers have higher education.
- Wealth quintile: there is no steady trend from poorest to richest. The "Richer" group is highest (10.0%), and "Middle" and "Richest" are lowest (6.8%).
- Antenatal care checks (BP, urine, blood, iron tablets): mothers who had these checks show slightly higher LBW rates (about 7.8–8.1% vs 6.7–7.5%). This is likely because mothers who get more care are also more likely to give birth in a facility and have their baby weighed.

### Region
- LBW rates vary widely across the 31 regions. The highest are Ruvuma and Kaskazini Unguja (around 14%), and the lowest are Rukwa, Kagera (3.3%) and Tanga (all under about 4%). Most regions sit fairly close to the overall 7.7%.
- Each region has only about 120–210 births, so some of this variation is likely due to chance. Region is for EDA only and will not be used in the model.
###

- Only babies who were weighed are included, and these are mostly babies born in health facilities. 

# Summary numbers

In [ ]:
#summary numbers
num_cols = ["maternal_age", "prior_live_births", "preceding_interval_m",
            "anc_visits", "first_anc_month"]

df[num_cols].describe().round(1)

In [ ]:
#histogram
df[num_cols].hist(bins=30, figsize=(12, 7))
plt.tight_layout()
plt.show()

Each chart shows how the values are spread, for example whether most mothers are in their 20s.

In [ ]:
# LBW rate by group
df["age_group"] = pd.cut(df["maternal_age"], bins=[14, 19, 24, 29, 34, 39, 49])
df["interval_group"] = pd.cut(df["preceding_interval_m"], bins=[0, 18, 24, 36, 60, 300])
df["births_group"] = pd.cut(df["prior_live_births"], bins=[-1, 0, 2, 4, 20],
                            labels=["0", "1-2", "3-4", "5+"])
df["anc_group"] = pd.cut(df["anc_visits"], bins=[-1, 0, 3, 7, 30],
                         labels=["0", "1-3", "4-7", "8+"])
df["first_anc_group"] = pd.cut(df["first_anc_month"], bins=[0, 3, 6, 9],
                               labels=["1-3 (early)", "4-6", "7-9 (late)"])

for col in ["age_group", "interval_group", "births_group", "anc_group", "first_anc_group"]:
    s = lbw_summary(col)
    display(Markdown(f"### {col}"))
    display(s)

    ax = s["lbw_rate_%"].plot(kind="barh", figsize=(6, 0.4 * len(s) + 1))
    ax.axvline(overall, linestyle="--", color="grey", label=f"Overall {overall:.1f}%")
    ax.set_xlabel("Weighted LBW rate (%)")
    ax.set_ylabel("")
    ax.legend()
    plt.show()

### Group cut-offs used

- Age: teenage mothers (under 20) and older mothers (35+) are often higher risk.
- Birth interval: under 18–24 months since the last birth is considered short.
- ANC visits: WHO recommends at least 8 visits; the older target was 4.
- First ANC month: months 1–3 is an early start; later is a late start.

### Numeric columns

- Maternal age: mothers are between 14 and 48, and the middle age is 26. There are more mothers at round ages like 25, 30 and 40, probably because some ages were rounded.
- Age and LBW: teenage mothers (15–19) have the most LBW babies (10.8%). It goes down to 6.3% for mothers aged 25–29, then goes up again for older mothers (8.7% at 40+).
- Birth interval: the middle gap since the last birth is 37 months. Babies born 18 months or less after the last birth have more LBW (10.0%) than those born 3–5 years after (5.9%).
- Prior live births: first-time mothers have more LBW (9.7%) than mothers with 1–4 children before (6.6–7.0%). It goes up a bit again for mothers with 5 or more (8.0%).
- ANC visits: most mothers had 4–7 visits, and they have the lowest LBW (7.1%). Mothers with 1–3 visits have the highest (9.3%). Only 150 mothers had 8 or more visits.
- First ANC month: not much difference between mothers who started early (7.3%) and late (6.6%).
- About 400 mothers have first_anc_month = 0. This is about the same as the number with no ANC visits.

## Conclusion

- 7.7% of babies in the Tanzania data were low birth weight.
- The biggest risk factor is twins (56.3% LBW).
- Teenage mothers, first-time mothers, short gaps between births, few ANC visits and high blood pressure are also linked to more LBW.
- Education, wealth and when ANC started don't show a clear pattern.
- Only weighed babies are included, so the data leans toward facility births.